# Enveda CASMI 2026: Tier-1 + Tier-2 Hybrid Pipeline

## Dual-Engine Molecule Identification from Mass Spectra
- **Tier 1 (Class 1 Candidates)**: High-speed row-group mass index scanning `train.parquet` + Numba square-root cosine dot product.
- **Tier 2 (Class 2 Candidates)**: Deep neural retrieval using `SpectrumFingerprintNet` (7.25M params) + in-silico fragmentation explainer over 422,926 COCONUT natural products.
- **Standardization**: Strict RDKit canonical tautomer `InChIKey14` deduplication (0 wasted ranking slots).
- **Kaggle Resource Compliance**: Runs in < 3 minutes on Kaggle CPU with < 2 GB RAM.

In [ ]:
"""Enveda CASMI 2026: Hybrid Pipeline (Tier 1 Library Matcher + Tier 2 COCONUT Neural Retrieval)."""
import os
import time
import numpy as np
import polars as pl
import pyarrow.parquet as pq
import pyarrow as pa
import torch
import torch.nn as nn
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Set
from numba import njit
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
_TAUTOMER_CANON = rdMolStandardize.TautomerCanonicalizer()
_MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# --- 1. ADDUCT MASS OFFSETS ---
ADDUCT_OFFSETS: Dict[str, float] = {
    "[M+H]+": 1.007276,
    "[M+NH4]+": 18.033823,
    "[M-H2O+H]+": -17.003289,
    "[M-2H2O+H]+": -35.013854,
    "[M+Na]+": 22.989218,
    "[M+K]+": 38.963158,
    "[M-H]-": -1.007276,
    "[M-H2O-H]-": -19.017841,
    "[M+CH2O2-H]-": 44.998203,
    "[M+Cl]-": 34.969402,
}

def calculate_neutral_mass(precursor_mz: float, adduct: str) -> Optional[float]:
    offset = ADDUCT_OFFSETS.get(adduct)
    return precursor_mz - offset if offset is not None else None

def smiles_to_inchikey14(smiles: str) -> str:
    if not smiles or not isinstance(smiles, str):
        return ""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return ""
        canon_mol = _TAUTOMER_CANON.canonicalize(mol)
        inchikey = Chem.MolToInchiKey(canon_mol)
        return inchikey.split("-")[0] if inchikey else ""
    except Exception:
        return ""

def smiles_to_fp(smiles: str) -> Optional[np.ndarray]:
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        fp = _MORGAN_GEN.GetFingerprint(mol)
        arr = np.zeros(2048, dtype=np.float32)
        for bit in fp.GetOnBits():
            arr[bit] = 1.0
        return arr
    except Exception:
        return None

# --- 2. NUMBA SPECTRAL COSINE MATCHER ---
@njit(fastmath=True)
def cosine_similarity_numba(
    mz1: np.ndarray,
    int1: np.ndarray,
    mz2: np.ndarray,
    int2: np.ndarray,
    mz_tol: float = 0.02,
) -> float:
    n1, n2 = len(mz1), len(mz2)
    if n1 == 0 or n2 == 0:
        return 0.0
    dot = 0.0
    norm1 = 0.0
    for i in range(n1):
        norm1 += int1[i] * int1[i]
    norm2 = 0.0
    for j in range(n2):
        norm2 += int2[j] * int2[j]
    if norm1 == 0.0 or norm2 == 0.0:
        return 0.0
    j_start = 0
    for i in range(n1):
        m1 = mz1[i]
        w1 = int1[i]
        while j_start < n2 and mz2[j_start] < m1 - mz_tol:
            j_start += 1
        best_diff = mz_tol
        best_w2 = 0.0
        j = j_start
        while j < n2 and mz2[j] <= m1 + mz_tol:
            diff = abs(mz2[j] - m1)
            if diff < best_diff:
                best_diff = diff
                best_w2 = int2[j]
            j += 1
        if best_w2 > 0.0:
            dot += w1 * best_w2
    denom = np.sqrt(norm1) * np.sqrt(norm2)
    return dot / denom if denom > 0.0 else 0.0

# --- 3. SPECTRUM FINGERPRINT NEURAL NET ---
def featurize_spectrum(mzs: np.ndarray, intensities: np.ndarray, precursor_mz: float, num_bins: int = 2000, max_mz: float = 1000.0) -> np.ndarray:
    bin_size = max_mz / num_bins
    frag_bins = np.zeros(num_bins, dtype=np.float32)
    loss_bins = np.zeros(num_bins, dtype=np.float32)
    w_ints = np.sqrt(np.maximum(intensities, 0.0))
    for mz, w in zip(mzs, w_ints):
        if 0 < mz < max_mz:
            b_idx = int(mz / bin_size)
            if b_idx < num_bins:
                frag_bins[b_idx] = max(frag_bins[b_idx], w)
        loss = precursor_mz - mz
        if 0 < loss < max_mz:
            l_idx = int(loss / bin_size)
            if l_idx < num_bins:
                loss_bins[l_idx] = max(loss_bins[l_idx], w)
    f_norm = np.linalg.norm(frag_bins)
    if f_norm > 0: frag_bins /= f_norm
    l_norm = np.linalg.norm(loss_bins)
    if l_norm > 0: loss_bins /= l_norm
    return np.concatenate([frag_bins, loss_bins, np.array([precursor_mz / max_mz], dtype=np.float32)])

class SpectrumFingerprintNet(nn.Module):
    def __init__(self, in_features: int = 4001, out_features: int = 2048, hidden_dim: int = 1024):
        super().__init__()
        self.block1 = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.GELU(), nn.Dropout(0.2))
        self.block2 = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.GELU(), nn.Dropout(0.2))
        self.head = nn.Sequential(nn.Linear(hidden_dim, out_features), nn.Sigmoid())
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h1 = self.block1(x)
        h2 = self.block2(h1) + h1
        return self.head(h2)

def continuous_tanimoto(pred_fp: np.ndarray, cand_fp: np.ndarray) -> float:
    dot = np.dot(pred_fp, cand_fp)
    denom = np.sum(pred_fp) + np.sum(cand_fp) - dot
    return float(dot / denom) if denom > 0 else 0.0

print("Modules initialized successfully!")


### Run Hybrid Retrieval Pipeline

In [ ]:
import sys
# Execute hybrid pipeline
from src.pipeline_v2 import run_hybrid_pipeline
t0 = time.time()
sub_df = run_hybrid_pipeline(
    test_parquet_path="data/test.parquet",
    train_parquet_path="data/train.parquet",
    coconut_parquet_path="data/external/coconut_indexed.parquet",
    model_weights_path="models/fingerprint_net.pt",
    output_path="data/submission_hybrid.csv",
)
print(f"Hybrid submission generated in {time.time() - t0:.1f}s!")
print(sub_df.head())
